# Ballon d'Or 2025/26 — goal contributions by opponent ranking

How strong were the opponents behind each candidate's numbers? One row per
player — headshot, name, G/A tally — and bars binned by **opponent rank band**
in steps of 15. Club opponents are ranked by their current global **Club Elo**
position; countries by the current **FIFA world ranking**.

**Viz by [@oladeeayo](https://github.com/oladeeayo)** — three 800 dpi PNGs in
[`figures/`](figures).

![Goals contribution by opponents ranking](figures/contribution_histograms.png)

**Contents** · [`goals_assists.csv`](goals_assists.csv) — every goal & assist by
the six candidates, Aug 2025 – Jul 2026 ·
[`club_ranks.csv`](club_ranks.csv) — Club Elo rank position per club opponent ·
[`players/`](players) — FotMob headshots · [`fonts/`](fonts) — DM Sans & Karla
(SIL OFL) · this notebook.


In [ ]:
from __future__ import annotations

import csv
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
from IPython.display import display

Image.MAX_IMAGE_PIXELS = None

HERE = Path.cwd()
assert (HERE / "goals_assists.csv").exists(), "run this notebook from its ballon-dor-contributions/ folder"
(HERE / "figures").mkdir(exist_ok=True)

# --- fonts ship with the repo --------------------------------------------
TITLE_FONT = "DM Sans"
BODY_FONT = "Karla"
import matplotlib.font_manager as fm
for _ttf in sorted((HERE / "fonts").glob("*.ttf")):
    fm.fontManager.addfont(str(_ttf))

# --- themes --------------------------------------------------------------
GREY_BG = "#F7F7F7"        # Opta Analyst canvas
SOC_BG = "#EFE9E6"         # soc-viz-of-the-week canvas
INK = "#1A1A1A"
SUB = "#4E616C"            # soc-viz subtitle colour
BAR_FC, BAR_EC = "#C9CDF4", "#6E77D4"       # club shade (hatched)
CTRY_FC, CTRY_EC = "#5A63C8", "#424BAF"     # country shade
BASE = "#B9B9C0"
TICK = "#4A4A52"
DPI = 800
WIDTH = 15                              # rank band width

# final 2026 Ballon d'Or ranking of our six players — rows follow it
BO_RANK = {"Harry Kane": 1, "Kylian Mbappé": 2, "Lamine Yamal": 3,
           "Ousmane Dembélé": 7, "Erling Haaland": 8, "Michael Olise": 9}

BINS = ["1-15", "16-30", "31-45", "46-60", "61-75", "76-90", "91-100", "100+"]


def bin_of(rank: int) -> int:
    if rank > 100:
        return len(BINS) - 1                       # 100+
    return min((rank - 1) // WIDTH, len(BINS) - 2)


## The data

`goals_assists.csv` has one row per goal or assist (378 events across the six
candidates) with the opponent and its kind. Country opponents carry their FIFA
rank directly; club opponents are looked up in `club_ranks.csv`, a two-column
extract of the current Club Elo table (**opponent → global rank position**,
1 = strongest) built for exactly this chart, so the notebook needs nothing else.

Rankings are a *current* snapshot — Club Elo and the FIFA list move — so a
re-render later can shift a club or two between bands.


In [ ]:
def club_ranks() -> dict[str, int | None]:
    """Club Elo rank position (1 = strongest) per club opponent."""
    out: dict[str, int | None] = {}
    with (HERE / "club_ranks.csv").open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            out[r["opponent"]] = int(r["club_rank"]) if r["club_rank"] else None
    return out


def gather():
    ranks = club_ranks()
    players: dict[str, dict] = {}
    with (HERE / "goals_assists.csv").open(newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            p = players.setdefault(r["player"], {
                "bins": [0] * len(BINS),      # pooled
                "cbins": [0] * len(BINS),     # club opponents only
                "fbins": [0] * len(BINS),     # country opponents only
                "goals": 0, "assists": 0, "n": 0})
            p["n"] += 1
            if r["event_type"] == "goal":
                p["goals"] += 1
            else:
                p["assists"] += 1
            if r["opponent_kind"] == "country":
                b = bin_of(int(r["opponent_fifa_rank"]))
                p["fbins"][b] += 1
            else:
                rank = ranks.get(r["opponent"])
                b = bin_of(rank) if rank else len(BINS) - 1   # unranked -> 100+
                p["cbins"][b] += 1
            p["bins"][b] += 1
    return sorted(players.items(), key=lambda kv: BO_RANK.get(kv[0], 999))


ordered = gather()
for name, p in ordered:
    club, ctry = sum(p["cbins"]), sum(p["fbins"])
    print(f"  {name:16s} {p['goals']:3d}G {p['assists']:3d}A  n={p['n']:3d} "
          f"(club {club} + country {ctry} = {club + ctry})")


## The chart

One row per player: headshot and name on the left, then the histogram — no
panel box, no gridlines, counts on top of the bars. In the stacked variant the
club segment sits below (hatched) and the country segment above it.


In [ ]:
def render(ordered, split: bool, bg: str, out_path: Path) -> None:
    n = len(ordered)
    ymax = max(max(p["bins"]) for _, p in ordered) + 5.6   # headroom for labels

    height = 3.2 * n + 3.6
    fig = plt.figure(figsize=(16.5, height), dpi=DPI, facecolor=bg)
    gs = fig.add_gridspec(n, 2, width_ratios=[1, 10.0],
                          left=0.055, right=0.972, top=0.822, bottom=0.075,
                          wspace=0.06, hspace=1.55)

    for i, (name, p) in enumerate(ordered):
        # ---- headshot at the far (left) end
        ax_img = fig.add_subplot(gs[i, 0])
        ax_img.set_axis_off()
        img_path = HERE / "players" / (re.sub(
            r"[^a-z0-9]+", "-",
            unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode().lower(),
        ).strip("-") + ".png")
        if img_path.exists():
            ax_img.imshow(Image.open(img_path).convert("RGBA"), aspect="equal")
        ax_img.text(0.5, -0.26, name, transform=ax_img.transAxes, ha="center",
                    va="top", fontsize=17, fontfamily=TITLE_FONT,
                    fontweight="bold", color=INK)
        ax_img.text(0.5, -0.50, f"{p['goals']} G | {p['assists']} A",
                    transform=ax_img.transAxes, ha="center", va="top",
                    fontsize=14, fontfamily=BODY_FONT, color=SUB)

        # ---- histogram row (no panel, no grid, counts on top)
        ax = fig.add_subplot(gs[i, 1])
        ax.set_facecolor("none")
        xs = range(len(BINS))
        if split:
            ax.bar(xs, p["cbins"], width=0.85, color=BAR_FC, edgecolor=BAR_EC,
                   linewidth=1.1, hatch="///", zorder=3)
            ax.bar(xs, p["fbins"], width=0.85, bottom=p["cbins"],
                   color=CTRY_FC, edgecolor=CTRY_EC, linewidth=1.1, zorder=3)
        else:
            ax.bar(xs, p["bins"], width=0.85, color=BAR_FC, edgecolor=BAR_EC,
                   linewidth=1.1, hatch="///", zorder=3)
        for x, v in enumerate(p["bins"]):
            if v:
                ax.text(x, v + 0.6, str(v), ha="center", va="bottom",
                        fontsize=17, fontfamily=BODY_FONT, fontweight="bold",
                        color=INK, zorder=4)

        ax.set_ylim(0, ymax)
        ax.set_xlim(-0.6, len(BINS) - 0.4)
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_color(BASE)
        ax.tick_params(axis="y", left=False, labelleft=False)
        ax.tick_params(axis="x", length=0, pad=10)
        ax.set_xticks(list(xs))
        ax.set_xticklabels(BINS, fontsize=16, fontfamily=BODY_FONT, color=TICK)

    # ---- header: centred DM Sans Bold title + two-line Karla subtitle
    title_txt = fig.text(0.5, 0.952, "GOALS CONTRIBUTION BY OPPONENTS RANKING",
                         fontsize=37, fontfamily=TITLE_FONT, fontweight="bold",
                         color=INK, ha="center", va="baseline")

    # ---- Ballon d'Or icon snug against the title's left side
    renderer = fig.canvas.get_renderer()
    bb = title_txt.get_window_extent(renderer=renderer)
    pxw, pxh = fig.get_size_inches() * fig.dpi
    icon = Image.open(HERE / "ballon_icon.png").convert("RGBA")
    ic_h = (bb.y1 - bb.y0) / pxh * 1.12
    ic_w = ic_h * icon.width / icon.height
    ax_icon = fig.add_axes([bb.x0 / pxw - 0.005 - ic_w,
                            (bb.y0 + bb.y1) / 2 / pxh - ic_h / 2, ic_w, ic_h])
    ax_icon.set_axis_off()
    ax_icon.imshow(icon)

    fig.text(0.5, 0.924, "2025/26 Ballon d'Or campaign | G+A against "
                         "opponents in each rank band",
             fontsize=17, fontfamily=BODY_FONT, color=SUB, ha="center", va="baseline")
    fig.text(0.5, 0.900, "Club opponents ranked by Club Elo, countries by FIFA "
                         "world rank | viz by @oladeeayo",
             fontsize=17, fontfamily=BODY_FONT, color=SUB, ha="center", va="baseline")

    if split:
        handles = [Patch(facecolor=CTRY_FC, edgecolor=CTRY_EC, label="Country"),
                   Patch(facecolor=BAR_FC, edgecolor=BAR_EC, hatch="///", label="Club")]
        fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.972, 0.878),
                   ncol=1, frameon=False, fontsize=15, labelcolor=INK,
                   labelspacing=0.6, handlelength=1.6, handleheight=1.1,
                   borderpad=0, borderaxespad=0)

    # ---- caption underneath + closing credit, both centred
    fig.text(0.5, 0.034, "Based on current Club Elo Ranking and FIFA ranking",
             fontsize=15, fontfamily=BODY_FONT, style="italic",
             color=SUB, ha="center", va="baseline")
    fig.text(0.5, 0.013, "Viz by Oladeeayo", fontsize=15, fontfamily=BODY_FONT,
             color=SUB, ha="center", va="baseline")

    fig.savefig(out_path, facecolor=bg, dpi=DPI)
    plt.close(fig)
    print(f"wrote {out_path.name}")


render(ordered, split=False, bg=GREY_BG, out_path=HERE / "figures" / "contribution_histograms.png")
render(ordered, split=True,  bg=GREY_BG, out_path=HERE / "figures" / "contribution_histograms_by_type.png")
render(ordered, split=False, bg=SOC_BG,  out_path=HERE / "figures" / "contribution_histograms_soc_bg.png")


In [ ]:
from IPython.display import Image as IPyImage

for name in ("contribution_histograms.png",
             "contribution_histograms_by_type.png",
             "contribution_histograms_soc_bg.png"):
    display(IPyImage(filename=str(HERE / "figures" / name), width=1050))


## What it shows

Rows follow the final 2026 Ballon d'Or order — Kane 1, Mbappé 2, Yamal 3,
Dembélé 7, Haaland 8, Olise 9.

* Kane's and Haaland's contributions skew towards the stronger clubs (low rank
  numbers), which is what you would expect from Bayern and City playing at the
  top of every table they enter.
* The **100+** band collects every opponent outside the top 100 — including
  the one club Club Elo does not track (Vendée Fontenay Foot).
* Country opponents are rarer per player but concentrated in qualifying and
  tournament play, so they cluster in bands 30–70.

**Coverage:** 81 club opponents + the national teams of the six players, across
378 goal/assist events, August 2025 – July 2026.

---

**Viz by [@oladeeayo](https://github.com/oladeeayo).**
Rankings: current Club Elo and the July 2026 FIFA world ranking.
Player images: FotMob. Fonts: DM Sans & Karla, SIL Open Font License.
